# Data Check: Lead-Lag Features

Quick visual check for EWLL lead-lag pipeline data quality.

1. Run the LeadLag pipeline stage to populate parquet data
2. Load and inspect the output features
3. Plot key features for visual inspection

## Setup

In [ ]:
import sys
sys.path.insert(0, '../../src')

import logging
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from makalu import settings as sys_env
from makalu.storage.parquet_store import ParquetStore
from qate.util.dt_range import DtRange

from pipeline.market_pair import EXECUTION_SIZE, LEAD_LAG_PAIRS
from lead_lag.lead_lag import LEAD_LAG_CONFIG_NEARHFT, LEAD_LAG_CONFIG_FAR

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
%matplotlib inline

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s %(message)s')

POPULATE_DATA = True

PAIR_KEY = 'eth'
MARKET_LEAD, MARKET_LAG = LEAD_LAG_PAIRS[PAIR_KEY]
DATA_START_DATE = '20260101'
DATA_END_DATE = '20260215'

DT_RANGE = DtRange.from_strings(DATA_START_DATE, DATA_END_DATE)
EXECUTION_SIZE_STR = EXECUTION_SIZE.get(MARKET_LAG.symbol)
CONFIG = LEAD_LAG_CONFIG_FAR

print(f'Leader:         {MARKET_LEAD.id}')
print(f'Target (lag):   {MARKET_LAG.id}')
print(f'Date range:     {DATA_START_DATE} - {DATA_END_DATE}')
print(f'Execution size: {EXECUTION_SIZE_STR}')

## 1. Run Pipeline

In [ ]:

from pipeline.framework import Pipeline
from pipeline.stages import LeadLagStage
from pipeline.lead_lag_steps import LeadLagStep

def populate_data():
    parquet_store = ParquetStore(sys_env.get_parquet_root_dir())

    pipeline = Pipeline(parquet_store)
    pipeline.stage(
        LeadLagStage(MARKET_LEAD, MARKET_LAG, EXECUTION_SIZE_STR)
        .step(LeadLagStep(CONFIG))
    )

    print('Running pipeline...')
    pipeline.process(DT_RANGE)
    print('Done.')

if POPULATE_DATA:
    populate_data()

## 2. Load & Inspect Parquet Data

In [ ]:
store = ParquetStore(sys_env.get_parquet_root_dir())
ll_store = store.create_lead_lag(MARKET_LEAD, MARKET_LAG, CONFIG.param_id())
df = ll_store.read(DT_RANGE)

# --- Time-window inspection controls ---
# Set WINDOW_MINUTES and WINDOW_INDEX to slice any Nth 5-min (or N-min) chunk.
# WINDOW_INDEX=0 → first window, WINDOW_INDEX=1 → second, etc.
# Or set INSPECT_START / INSPECT_END directly (overrides window if both are set).
WINDOW_MINUTES = 1440 * 46
WINDOW_INDEX   = 0          # change this to jump to the Nth window

# Derive start/end from the df's actual time range so it works for any date
df['_dt'] = pd.to_datetime(df['ts'], unit='s')
day_start = df['_dt'].min().floor('min')   # first timestamp in the loaded data

INSPECT_START = day_start + pd.Timedelta(minutes=WINDOW_MINUTES * WINDOW_INDEX)
INSPECT_END   = INSPECT_START + pd.Timedelta(minutes=WINDOW_MINUTES)

print(f'Full data   : {df["_dt"].min()}  →  {df["_dt"].max()}  ({len(df):,} rows)')
print(f'Shape       : {df.shape}')
print(f'Columns     : {list(df.columns)}')

# --- Slice the window ---
mask   = (df['_dt'] >= INSPECT_START) & (df['_dt'] < INSPECT_END)
df = df.loc[mask].drop(columns=['_dt'])

# print(f'\nWindow [{WINDOW_INDEX}]  : {INSPECT_START}  →  {INSPECT_END}  ({len(df_win):,} rows)')
# print(f'NaN counts  :\n{df_win.isna().sum()}')
# display(df_win.describe())
# display(df_win.head(10))


## 3. Plot Key Features

In [ ]:
# Convert ts to datetime for plotting
df['datetime'] = pd.to_datetime(df['ts'], unit='s')

# Find all max_corr columns
corr_cols = [c for c in df.columns if c.startswith('ll_max_corr')]
lag_cols = [c for c in df.columns if c.startswith('ll_opt_lag_ms')]
gain_cols = [c for c in df.columns if c.startswith('ll_alpha_gain_log')]
stability_cols = [c for c in df.columns if c.startswith('ll_lag_stability_lb')]
drift_cols = [c for c in df.columns if c.startswith('ll_drift')]

print(f'Max corr columns: {corr_cols}')
print(f'Opt lag columns:  {lag_cols}')
print(f'Alpha gain cols:  {gain_cols}')
print(f'Stability cols:   {stability_cols}')
print(f'Drift cols:       {drift_cols}')


In [ ]:
# Plot ll_max_corr
# ll_max_corr is the peak exponentially-weighted cross-correlation between the leader's and lag market's log-returns, found by scanning over a grid of time offsets.
#
# In plain terms:
#
# - The algorithm shifts the leader's return series backwards by k steps (for each k in search_lags_ms = range(0, 5100, 100)) and computes the EW correlation with the lag market's returns at each shift.
# - ll_max_corr is the highest absolute correlation found across all those shifts — i.e. max |ρ(k)| over the lag grid.
# - It's clipped to [-1, 1] and its sign is preserved (so negative values mean the leader's move predicts an opposite move in the lag market).
#
# The full column name is ll_max_corr_{window_sec}s_{span_sec}s, e.g. ll_max_corr_10s_5s, where:
# - window_sec — how many seconds of return history were used (the sub-window: 10s, 20s, 30s, or 60s)
# - span_sec — the EW decay span in seconds that controls how much weight recent data gets (5s, 15s, or 30s)
#
# So a high ll_max_corr (close to 1) means the leader is strongly predictive of the lag market at some lag offset. A value near 0 means no detectable lead-lag relationship in that window.

import matplotlib.dates as mdates

if corr_cols:
    fig, axes = plt.subplots(len(corr_cols), 1, figsize=(40, 3 * len(corr_cols)), sharex=True)
    if len(corr_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, corr_cols):
        ax.plot(df['datetime'], df[col], linewidth=0.5)
        ax.set_ylabel(col, fontsize=8)
        ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
        ax.tick_params(axis='x', rotation=45)
    axes[0].set_title(f'{MARKET_LEAD.id} -> {MARKET_LAG.id} Max Correlation')
    axes[0].xaxis.set_major_locator(mdates.DayLocator(interval=1))
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot ll_opt_lag_ms (normalized)
# ll_opt_lag_ms is the time offset (in ms) at which ll_max_corr was achieved — i.e. the lag that produced the peak correlation — but normalized to [0, 1] by dividing by max_lag_ms (5100ms).
#
# opt_lag_ms = float(opt_lag_idx * config.resample_ms)   # actual ms: 0, 100, 200, ... 5000
# norm_lag   = opt_lag_ms / config.max_lag_ms             # normalized: 0.0 → 1.0
# fields[f"ll_opt_lag_ms{suffix}"] = norm_lag
#
# So concretely:
# - 0.0 → the leader and lag move simultaneously (0ms offset gives best correlation)
# - 1.0 → the leader leads the lag by 5000ms (5 seconds), the maximum searched
# - 0.2 → optimal lead time is ~1000ms (1 second)

if lag_cols:
    fig, axes = plt.subplots(len(lag_cols), 1, figsize=(40, 3 * len(lag_cols)), sharex=True)
    if len(lag_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, lag_cols):
        ax.plot(df['datetime'], df[col], linewidth=0.5, color='orange')
        ax.set_ylabel(col, fontsize=8)
    axes[0].set_title(f'{MARKET_LEAD.id} -> {MARKET_LAG.id} Optimal Lag (normalized)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot ll_alpha_gain
# ll_alpha_gain is the ratio of the peak correlation to the zero-lag correlation — i.e. how much better the optimal lag is compared to no lag at all.

# - 1.0 → shifting the leader by its optimal lag gives no improvement over zero-lag; the leader isn't really "leading", they just move together
# - 100.0 (max, clipped) → the optimal-lag correlation is much stronger than the zero-lag correlation, meaning the leader genuinely predicts the lag market ahead of time
# - e.g. 3.0 → the signal is 3× stronger when you apply the optimal time shift

# Intuitively it answers: "Is the leader actually leading, or just correlated in real-time?" A high ll_alpha_gain with a non-zero ll_opt_lag_ms is the clearest signal that a true lead-lag relationship exists and is
# exploitable.

# - ll_alpha_gain is clipped to [1.0, 100.0], so it can spike to 100 when corr_at_zero is near zero. That large dynamic range can dominate ML training.
# - ll_alpha_gain_log compresses that range: log1p(1.0) ≈ 0.69, log1p(100.0) ≈ 4.62 — much more ML-friendly as a feature.

if gain_cols:
    fig, axes = plt.subplots(len(gain_cols), 1, figsize=(40, 3 * len(gain_cols)), sharex=True)
    if len(gain_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, gain_cols):
        ax.plot(df['datetime'], df[col], linewidth=0.5, color='green')
        ax.set_ylabel(col, fontsize=8)
        ax.axhline(y=1, color='red', linewidth=0.5, linestyle='--')
    axes[0].set_title(f'{MARKET_LEAD.id} -> {MARKET_LAG.id} Alpha Gain')
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot ll_lag_stability_lb{N} and ll_drift
# ll_lag_stability_lb{N} is the log1p of the rolling std of ll_opt_lag_ms over the last N windows
# (configured via stability_lookbacks in LeadLagConfig) — i.e. how much the optimal lag has been
# jumping around recently. Multiple lb columns appear when stability_lookbacks has multiple values.
#
# - 0.0 → perfectly stable (same lag every window for the last N steps)
# - higher → erratic / unstable, changing significantly from window to window
#
# Practically: "Can I trust the current optimal lag?" Low = reliable signal, High = noisy.

if stability_cols:
    fig, axes = plt.subplots(len(stability_cols), 1, figsize=(40, 3 * len(stability_cols)), sharex=True)
    if len(stability_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, stability_cols):
        ax.plot(df['datetime'], df[col], linewidth=0.5, color='purple')
        ax.set_ylabel(col, fontsize=8)
    axes[0].set_title(f'{MARKET_LEAD.id} -> {MARKET_LAG.id} Lag Stability (log1p)')
    plt.tight_layout()
    plt.show()


# ll_drift is the change in optimal lag over the last 5 seconds — i.e. is the lead time getting longer or shorter recently.

# - 0.0 → the optimal lag is not shifting, stable relationship
# - +1.0 → the optimal lag has increased by the maximum (5000ms) over the last 5 seconds — the leader is taking longer and longer to influence the lag market
# - -1.0 → the optimal lag has decreased by the maximum — the two markets are converging in time, leader influence is becoming more immediate

# With window_step_sec=1, drift_lookback=5 means it compares the current optimal lag against what it was 5 windows ago (5 seconds ago).

# Practically it answers: "Is the lead-lag relationship drifting?" For trading, a drifting lag might mean the relationship is transitioning — either strengthening/tightening (negative drift) or weakening/elongating
# (positive drift). Combined with ll_lag_stability_lb{N}, a low-stability + high-drift situation is a warning that the signal is in flux and unreliable.

if drift_cols:
    fig, axes = plt.subplots(len(drift_cols), 1, figsize=(40, 3 * len(drift_cols)), sharex=True)
    if len(drift_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, drift_cols):
        ax.plot(df['datetime'], df[col], linewidth=0.5, color='brown')
        ax.set_ylabel(col, fontsize=8)
        ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
        ax.axhline(y=1, color='red', linewidth=0.5, linestyle=':')
        ax.axhline(y=-1, color='red', linewidth=0.5, linestyle=':')
    axes[0].set_title(f'{MARKET_LEAD.id} -> {MARKET_LAG.id} Lag Drift (normalized)')
    plt.tight_layout()
    plt.show()


## 4. Correlation Heatmap

In [ ]:
# Correlation heatmap across all lead-lag features
ll_cols = [c for c in df.columns if c.startswith('ll_')]
if ll_cols:
    fig, ax = plt.subplots(figsize=(max(10, len(ll_cols) * 0.6), max(8, len(ll_cols) * 0.5)))
    corr_matrix = df[ll_cols].corr()
    sns.heatmap(corr_matrix, annot=False, cmap='RdBu_r', center=0, ax=ax,
                xticklabels=True, yticklabels=True)
    ax.set_title('Lead-Lag Feature Correlation Heatmap')
    plt.xticks(fontsize=6, rotation=90)
    plt.yticks(fontsize=6)
    plt.tight_layout()
    plt.show()

## 5. Data Consistency Summary

In [ ]:
print('=' * 60)
print('Data Consistency Summary')
print('=' * 60)

print(f'\n--- LeadLag Features ---')
print(f'  Leader:     {MARKET_LEAD.id}')
print(f'  Target:     {MARKET_LAG.id}')
print(f'  Rows:       {len(df):,}')
print(f'  Columns:    {len(df.columns)}')
ts_min = pd.to_datetime(df['ts'].min(), unit='s') if not df.empty else 'N/A'
ts_max = pd.to_datetime(df['ts'].max(), unit='s') if not df.empty else 'N/A'
print(f'  Time range: {ts_min} ~ {ts_max}')
total_nan = df.isna().sum().sum()
print(f'  Total NaN:  {total_nan:,} ({total_nan / max(df.size, 1) * 100:.2f}%)')
dupes = df.duplicated(subset=['ts']).sum()
print(f'  Duplicate timestamps: {dupes}')

# Sanity checks on correlation values
print(f'\n--- Sanity Checks ---')
for col in corr_cols:
    vals = df[col].dropna()
    in_range = ((vals >= -1) & (vals <= 1)).all()
    print(f'  {col} in [-1, 1]: {in_range}')

for col in lag_cols:
    vals = df[col].dropna()
    in_range = ((vals >= 0) & (vals <= 1)).all()
    print(f'  {col} in [0, 1]: {in_range}')

for col in drift_cols:
    vals = df[col].dropna()
    in_range = ((vals >= -1) & (vals <= 1)).all()
    print(f'  {col} in [-1, 1]: {in_range}')